# 03 -- Conversation Dynamics

Analyses sentiment trajectories within discussion threads to uncover
how tone evolves, escalates, or stabilises across a conversation.

Pipeline:
1. Load and validate the sentiment-enriched tweet data
2. Compute per-tweet rolling features (delta, rolling mean/std)
3. Aggregate per-thread summary metrics (arc type, trend slope, changepoints, emotion profile)
4. Visualise thread trajectories, arc distributions, and emotion heatmaps

**Input:** `DATA_DIR/sentiment_results.csv` (output of `02_sentiment_emotion.ipynb`)  
**Output:** `DATA_DIR/tweet_level_dynamics.csv`, `DATA_DIR/thread_summaries.csv`, plots in `PLOT_DIR/`


## Dependencies

In [ ]:
# Run once -- safe to skip if already installed
# !pip install ruptures


## Imports

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings('ignore')


## Config

All tunable thresholds are in one place so that re-running with a different
simulation only requires changing this cell.


In [ ]:
DATA_DIR = "data"   # <- change to your local path
PLOT_DIR = f"{DATA_DIR}/plots"

INPUT_CSV        = f"{DATA_DIR}/sentiment_results.csv"
OUTPUT_TWEETS    = f"{DATA_DIR}/tweet_level_dynamics.csv"
OUTPUT_SUMMARIES = f"{DATA_DIR}/thread_summaries.csv"

# Primary continuous sentiment signal in [-1, +1]
DEFAULT_SIGNAL   = "roberta_compound"
SECONDARY_SIGNAL = "vader_compound"

# Emotion axes
EMOTION_SIGNALS = ["emotion_valence", "emotion_arousal"]
EMOTION_CLASSES = ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"]

# Thread filtering
MIN_TURNS_DYNAMICS = 3      # threads shorter than this -> flagged, not analysed

# Arc classification thresholds
SLOPE_THRESH = 0.05         # |slope| < this -> "stable"
VOL_THRESH   = 0.25         # std > this -> "chaotic"

# Change-point detection (ruptures PELT)
CPD_MODEL   = "rbf"         # kernel: "rbf" | "l2" | "l1"
CPD_PENALTY = 3             # lower = more breakpoints

# Plot settings
PLOT_STYLE     = "seaborn-v0_8-whitegrid"
CMAP_DIVERGING = "RdYlGn"
CMAP_EMOTION   = "YlOrRd"
FIG_DPI        = 150

Path(PLOT_DIR).mkdir(parents=True, exist_ok=True)


## Load data

In [ ]:
df_raw = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df_raw):,} rows x {len(df_raw.columns)} columns")
df_raw.head()


## Load & validate

Sorts by thread and turn order, adds `turn` (0-based index within each thread),
`thread_len`, and `turn_rel` (relative position in [0, 1]) which enables
comparisons across threads of different lengths.


In [ ]:
def load_and_validate(df: pd.DataFrame, sort_col: str) -> pd.DataFrame:
    """
    Validate required columns, sort by thread and turn order, and enrich with:
      turn        -- 0-based turn index within each thread
      thread_len  -- total turns in the thread
      turn_rel    -- relative position in [0, 1]
    """
    print(f"[load] Validating input DataFrame ...")
    print(f"  -> {len(df):,} rows x {len(df.columns)} cols")

    required = {"thread_id", sort_col}
    missing  = required - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    for col in [DEFAULT_SIGNAL, SECONDARY_SIGNAL, *EMOTION_SIGNALS]:
        if col not in df.columns:
            print(f"  Warning: column '{col}' not found -- will be skipped.")

    df = df.sort_values(["thread_id", sort_col]).reset_index(drop=True)
    df["turn"]       = df.groupby("thread_id").cumcount()
    df["thread_len"] = df.groupby("thread_id")["turn"].transform("max") + 1
    df["turn_rel"]   = df["turn"] / (df["thread_len"] - 1).clip(lower=1)

    short = df[df["thread_len"] < MIN_TURNS_DYNAMICS]["thread_id"].nunique()
    total = df["thread_id"].nunique()
    print(f"  -> {total:,} threads total | "
          f"{short:,} with < {MIN_TURNS_DYNAMICS} turns (flagged, excluded from dynamics)\n")
    return df


df = load_and_validate(df_raw, sort_col="round")


## Tweet-level rolling features

For each tweet, computes within-thread rolling statistics on the primary
sentiment signal. `sentiment_agreement` flags tweets where VADER and
RoBERTa disagree by more than 0.4 -- useful for identifying ambiguous text.


In [ ]:
def compute_tweet_level_features(df: pd.DataFrame, signal: str) -> pd.DataFrame:
    """
    Add within-thread rolling features for a given sentiment signal.

    New columns:
      {signal}_delta       -- change from previous turn
      {signal}_roll3_mean  -- 3-turn rolling mean (smoothed signal)
      {signal}_roll3_std   -- 3-turn rolling std (local volatility)
      sentiment_agreement  -- |roberta_compound - vader_compound| (disagreement proxy)
      low_agreement        -- True when disagreement > 0.4
    """
    if signal not in df.columns:
        print(f"  [tweet features] Signal '{signal}' missing -- skipping.")
        return df

    print(f"[tweet-level] Computing rolling features on '{signal}' ...")
    df = df.copy().sort_values(["thread_id", "turn"])
    grp = df.groupby("thread_id")[signal]

    df[f"{signal}_delta"]      = grp.diff()
    df[f"{signal}_roll3_mean"] = grp.transform(lambda s: s.rolling(3, min_periods=1).mean())
    df[f"{signal}_roll3_std"]  = grp.transform(lambda s: s.rolling(3, min_periods=2).std())

    if DEFAULT_SIGNAL in df.columns and SECONDARY_SIGNAL in df.columns:
        df["sentiment_agreement"] = (df[DEFAULT_SIGNAL] - df[SECONDARY_SIGNAL]).abs()
        df["low_agreement"]       = df["sentiment_agreement"] > 0.4

    print("  -> done.\n")
    return df


df = compute_tweet_level_features(df, signal=DEFAULT_SIGNAL)


## Thread-level summary metrics

Aggregates each thread into a single row capturing:
- Trend slope and R^2 (OLS fit over turn indices)
- Arc type classification (stable / escalating / de-escalating / chaotic / V-shape / inverted-V)
- Changepoint detection via PELT (ruptures library)
- Dominant emotion and Shannon entropy over the emotion distribution
- Mean valence and arousal axes


In [ ]:
def _linear_trend(values: np.ndarray):
    """Return (slope, r_squared) of OLS fit over turn indices."""
    if len(values) < 2:
        return np.nan, np.nan
    x = np.arange(len(values), dtype=float)
    res = stats.linregress(x, values)
    return res.slope, res.rvalue ** 2


def _detect_changepoints(values: np.ndarray, penalty: float = CPD_PENALTY) -> list:
    """
    Use PELT (Pruned Exact Linear Time) to find structural break turns.
    Returns list of turn indices where a change point was detected.
    Returns [] if ruptures is not installed or the series is too short.
    """
    try:
        import ruptures as rpt
    except ImportError:
        return []
    if len(values) < 4:
        return []
    try:
        algo = rpt.Pelt(model=CPD_MODEL, min_size=2, jump=1).fit(
            values.reshape(-1, 1).astype(float))
        breaks = algo.predict(pen=penalty)
        return [b - 1 for b in breaks if b < len(values)]
    except Exception:
        return []


def _arc_type(slope: float, volatility: float, n_changepoints: int) -> str:
    """
    Classify the emotional arc of a thread into one of six types:
      stable        -- low volatility, negligible slope
      escalating    -- consistently rising sentiment
      de-escalating -- consistently falling sentiment
      chaotic       -- high volatility regardless of slope
      V-shape       -- one changepoint, overall positive slope
      inverted-V    -- one changepoint, overall negative slope
    """
    if pd.isna(slope) or pd.isna(volatility):
        return "unknown"
    if volatility > VOL_THRESH:
        return "chaotic"
    if n_changepoints == 1:
        return "V-shape" if slope > 0 else "inverted-V"
    if abs(slope) < SLOPE_THRESH:
        return "stable"
    return "escalating" if slope > 0 else "de-escalating"


def compute_thread_summaries(df: pd.DataFrame, signal: str) -> pd.DataFrame:
    """
    One row per thread. Key columns:
      thread_id, thread_len, short_thread
      mean_sentiment, std_sentiment, trend_slope, trend_r2
      sentiment_start, sentiment_end, sentiment_delta, arc_type
      n_changepoints, changepoint_turns, first_changepoint_turn
      dominant_emotion, emotion_entropy
      emotion_valence_mean, emotion_arousal_mean
      mean_low_agreement
    """
    print(f"[thread summaries] Aggregating {df['thread_id'].nunique():,} threads ...")
    records = []

    for tid, grp in df.groupby("thread_id"):
        grp = grp.sort_values("turn")
        n   = len(grp)
        rec = {"thread_id": tid, "thread_len": n,
               "short_thread": n < MIN_TURNS_DYNAMICS}

        if signal in grp.columns and grp[signal].notna().sum() >= 2:
            vals = grp[signal].dropna().values
            slope, r2 = _linear_trend(vals)
            rec.update({
                "mean_sentiment":  float(np.mean(vals)),
                "std_sentiment":   float(np.std(vals)),
                "trend_slope":     float(slope),
                "trend_r2":        float(r2),
                "sentiment_start": float(vals[0]),
                "sentiment_end":   float(vals[-1]),
                "sentiment_delta": float(vals[-1] - vals[0]),
                "arc_type":        _arc_type(slope, float(np.std(vals)),
                                             len(_detect_changepoints(vals))),
            })
            if n >= MIN_TURNS_DYNAMICS:
                cps = _detect_changepoints(vals)
                rec["n_changepoints"]         = len(cps)
                rec["changepoint_turns"]      = str(cps)
                rec["first_changepoint_turn"] = cps[0] if cps else np.nan
            else:
                rec.update({"n_changepoints": np.nan,
                             "changepoint_turns": np.nan,
                             "first_changepoint_turn": np.nan})
        else:
            rec.update({k: np.nan for k in [
                "mean_sentiment","std_sentiment","trend_slope","trend_r2",
                "sentiment_start","sentiment_end","sentiment_delta","arc_type",
                "n_changepoints","changepoint_turns","first_changepoint_turn"]})

        emotion_cols = [f"emotion_{e}" for e in EMOTION_CLASSES
                        if f"emotion_{e}" in grp.columns]
        if emotion_cols:
            emotion_means = grp[emotion_cols].mean()
            dominant = emotion_means.idxmax().replace("emotion_", "")
            probs = emotion_means.values
            probs = probs / probs.sum() if probs.sum() > 0 else probs
            rec["dominant_emotion"] = dominant
            rec["emotion_entropy"]  = float(-np.sum(probs * np.log(probs + 1e-9)))
            for col in emotion_cols:
                rec[f"{col}_mean"] = float(grp[col].mean())

        for ax in EMOTION_SIGNALS:
            if ax in grp.columns:
                rec[f"{ax}_mean"] = float(grp[ax].mean())

        if "low_agreement" in grp.columns:
            rec["mean_low_agreement"] = float(grp["low_agreement"].mean())

        records.append(rec)

    summary = pd.DataFrame(records)
    print(f"  -> {len(summary):,} thread summaries computed.")
    if "arc_type" in summary.columns:
        arc_counts = summary["arc_type"].value_counts()
        print("  Arc distribution:")
        for arc, cnt in arc_counts.items():
            print(f"    {arc:<20} {cnt:>5}  ({100 * cnt / len(summary):.1f}%)")
    print()
    return summary


summary = compute_thread_summaries(df, signal=DEFAULT_SIGNAL)


## Visualisations

Six plots covering:
- Sample thread trajectories by arc type
- Arc type distribution
- Emotion heatmap per arc type
- Average sentiment over relative thread position
- Valence-arousal scatter (Russell's circumplex)
- Changepoint timing histogram


In [ ]:
def _safe_style():
    try:
        plt.style.use(PLOT_STYLE)
    except Exception:
        plt.style.use("ggplot")


ARC_COLORS = {
    "stable":        "#4C9BE8",
    "escalating":    "#2ECC71",
    "de-escalating": "#E74C3C",
    "chaotic":       "#F39C12",
    "V-shape":       "#9B59B6",
    "inverted-V":    "#1ABC9C",
    "unknown":       "#CCCCCC",
}


def plot_sample_threads(df, summary, signal, out_dir, n_per_arc=2):
    """Grid of sample threads (one panel each), coloured by arc type."""
    if signal not in df.columns:
        return
    _safe_style()
    out_dir = Path(out_dir)

    arc_types  = [a for a in summary["arc_type"].dropna().unique() if a != "unknown"]
    sample_ids = []
    for arc in arc_types:
        sample_ids.extend(
            summary[summary["arc_type"] == arc]["thread_id"].head(n_per_arc).tolist())

    if not sample_ids:
        return

    ncols = 3
    nrows = int(np.ceil(len(sample_ids) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 3.5 * nrows), squeeze=False)
    fig.suptitle(f"Sample Thread Sentiment Trajectories  [{signal}]",
                 fontsize=14, fontweight="bold", y=1.01)

    for idx, tid in enumerate(sample_ids):
        ax  = axes[idx // ncols][idx % ncols]
        grp = df[df["thread_id"] == tid].sort_values("turn")
        row = summary[summary["thread_id"] == tid].iloc[0]
        arc = row.get("arc_type", "unknown")
        color = ARC_COLORS.get(arc, "#888888")

        turns = grp["turn"].values
        raw   = grp[signal].values
        roll_col = f"{signal}_roll3_mean"

        ax.plot(turns, raw, "o--", color=color, alpha=0.6,
                linewidth=1.2, markersize=4, label="raw")
        if roll_col in grp.columns:
            ax.plot(turns, grp[roll_col].values, "-",
                    color=color, linewidth=2.2, label="roll-3")

        try:
            cps = eval(str(row.get("changepoint_turns", "[]")))
        except Exception:
            cps = []
        for cp in cps:
            ax.axvline(cp, color="black", linestyle=":", linewidth=1, alpha=0.7)

        if len(turns) >= 2 and not pd.isna(row.get("trend_slope")):
            x_fit = np.array([turns[0], turns[-1]], dtype=float)
            y_fit = row["sentiment_start"] + row["trend_slope"] * x_fit
            ax.plot(x_fit, y_fit, "--", color="grey", linewidth=1, alpha=0.5)

        ax.axhline(0, color="grey", linewidth=0.8, alpha=0.4)
        ax.set_ylim(-1.1, 1.1)
        ax.set_title(f"Thread {tid}  [{arc}]  n={len(grp)}", fontsize=9)
        ax.set_xlabel("Turn", fontsize=8)
        ax.set_ylabel(signal, fontsize=8)
        ax.legend(fontsize=7, loc="lower right")

    for idx in range(len(sample_ids), nrows * ncols):
        axes[idx // ncols][idx % ncols].set_visible(False)

    plt.tight_layout()
    out = out_dir / "sample_thread_trajectories.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {out}")


def plot_arc_distribution(summary, out_dir):
    """Bar chart of arc type distribution."""
    if "arc_type" not in summary.columns:
        return
    _safe_style()
    counts = summary["arc_type"].value_counts()
    colors = list(ARC_COLORS.values())

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(counts.index, counts.values,
                  color=colors[:len(counts)], edgecolor="white", linewidth=0.8)
    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.5, str(val),
                ha="center", va="bottom", fontsize=9)
    ax.set_title("Thread Arc Type Distribution", fontsize=13, fontweight="bold")
    ax.set_xlabel("Arc Type")
    ax.set_ylabel("Number of Threads")
    plt.tight_layout()
    out = Path(out_dir) / "arc_distribution.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {out}")


def plot_emotion_heatmap(summary, out_dir):
    """Heatmap: average emotion probability per arc type."""
    emotion_cols = [f"emotion_{e}_mean" for e in EMOTION_CLASSES]
    available = [c for c in emotion_cols if c in summary.columns]
    if not available or "arc_type" not in summary.columns:
        print("  [plot] Emotion heatmap skipped -- columns not available.")
        return
    _safe_style()
    heat = (summary.groupby("arc_type")[available]
                   .mean()
                   .rename(columns=lambda c: c.replace("emotion_", "").replace("_mean", "")))

    fig, ax = plt.subplots(figsize=(10, max(3, len(heat) * 0.8 + 1)))
    sns.heatmap(heat, annot=True, fmt=".2f", cmap=CMAP_EMOTION,
                linewidths=0.5, ax=ax, cbar_kws={"label": "mean prob"})
    ax.set_title("Average Emotion Profile per Arc Type", fontsize=13, fontweight="bold")
    ax.set_xlabel("Emotion")
    ax.set_ylabel("Arc Type")
    plt.tight_layout()
    out = Path(out_dir) / "emotion_heatmap_by_arc.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {out}")


def plot_sentiment_over_relative_position(df, signal, out_dir):
    """Average sentiment at each relative position bin [0, 1] across all threads."""
    if signal not in df.columns or "turn_rel" not in df.columns:
        return
    _safe_style()
    df2 = df[df["thread_len"] >= MIN_TURNS_DYNAMICS].copy()
    df2["pos_bin"] = pd.cut(df2["turn_rel"], bins=10,
                             labels=[f"{i/10:.1f}" for i in range(10)])
    agg = df2.groupby("pos_bin")[signal].agg(["mean", "std"]).reset_index()

    fig, ax = plt.subplots(figsize=(9, 4))
    x = np.arange(len(agg))
    ax.plot(x, agg["mean"], "o-", color="#4C9BE8", linewidth=2, markersize=6)
    ax.fill_between(x, agg["mean"] - agg["std"], agg["mean"] + agg["std"],
                    color="#4C9BE8", alpha=0.15, label="+-1 std")
    ax.axhline(0, color="grey", linewidth=0.8, alpha=0.5, linestyle="--")
    ax.set_xticks(x)
    ax.set_xticklabels(["Start","","","","","","","","","End"], fontsize=9)
    ax.set_ylim(-1, 1)
    ax.set_title("Average Sentiment Across Relative Thread Position
"
                 "(0 = first turn, 1 = last turn)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Relative position in thread")
    ax.set_ylabel(signal)
    ax.legend(fontsize=9)
    plt.tight_layout()
    out = Path(out_dir) / "sentiment_over_thread_position.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {out}")


def plot_valence_arousal_by_arc(summary, out_dir):
    """Scatter of mean valence vs mean arousal, coloured by arc type."""
    x_col, y_col = "emotion_valence_mean", "emotion_arousal_mean"
    if x_col not in summary.columns or y_col not in summary.columns:
        print("  [plot] Valence/arousal scatter skipped -- columns not available.")
        return
    _safe_style()
    fig, ax = plt.subplots(figsize=(7, 6))
    for arc, grp in summary.groupby("arc_type"):
        ax.scatter(grp[x_col], grp[y_col], label=arc, alpha=0.55, s=20,
                   color=ARC_COLORS.get(arc, "#888"))
    ax.axhline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)
    ax.axvline(0, color="grey", linewidth=0.8, linestyle="--", alpha=0.5)
    for xp, yp, lbl in [( 0.5, 0.5, "joy/excitement"),
                          (-0.5, 0.5, "anger/fear"),
                          (-0.5,-0.5, "sadness/boredom"),
                          ( 0.5,-0.5, "calm/content")]:
        ax.text(xp, yp, lbl, ha="center", va="center", fontsize=8, color="grey", alpha=0.7)
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1)
    ax.set_xlabel("Valence  (joy <-> negative affect)", fontsize=10)
    ax.set_ylabel("Arousal  (activated <-> deactivated)", fontsize=10)
    ax.set_title("Valence-Arousal Space by Arc Type
(Russell's Circumplex)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=8, loc="lower right")
    plt.tight_layout()
    out = Path(out_dir) / "valence_arousal_by_arc.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {out}")


def plot_changepoint_timing(summary, out_dir):
    """Histogram of first changepoint relative position -- when do tone shifts happen?"""
    col = "first_changepoint_turn"
    if col not in summary.columns:
        return
    _safe_style()
    sub = summary[summary[col].notna() & summary["thread_len"].notna()].copy()
    if len(sub) == 0:
        return
    sub["cp_rel"] = sub[col] / (sub["thread_len"] - 1).clip(lower=1)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(sub["cp_rel"], bins=20, color="#E74C3C", edgecolor="white",
            linewidth=0.6, alpha=0.85)
    ax.axvline(sub["cp_rel"].median(), color="black", linestyle="--",
               linewidth=1.5, label=f"median={sub['cp_rel'].median():.2f}")
    ax.set_xlabel("Relative position of first changepoint", fontsize=10)
    ax.set_ylabel("Number of threads", fontsize=10)
    ax.set_title("When Do Tone Shifts Happen?
(First changepoint, relative position)",
                 fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    plt.tight_layout()
    out = Path(out_dir) / "changepoint_timing.png"
    fig.savefig(out, dpi=FIG_DPI, bbox_inches="tight")
    plt.show()
    print(f"  [saved] {out}")


# Run all plots
print("[plots] Generating figures ...")
plot_sample_threads(df, summary, DEFAULT_SIGNAL, PLOT_DIR)
plot_arc_distribution(summary, PLOT_DIR)
plot_emotion_heatmap(summary, PLOT_DIR)
plot_sentiment_over_relative_position(df, DEFAULT_SIGNAL, PLOT_DIR)
plot_valence_arousal_by_arc(summary, PLOT_DIR)
plot_changepoint_timing(summary, PLOT_DIR)
print("Done.")


## Save outputs

In [ ]:
df.to_csv(OUTPUT_TWEETS, index=False)
summary.to_csv(OUTPUT_SUMMARIES, index=False)
print(f"Tweet-level dynamics -> {OUTPUT_TWEETS}")
print(f"Thread summaries     -> {OUTPUT_SUMMARIES}")
